# 🎙️ VoiceBatch Studio v2.0.1 - Full Features Fixed
इसमें Upload Fix, Speed, Pitch और Realistic Voice बटन जोड़ दिए गए हैं।

In [ ]:
# @title 📥 Step 1: इंस्टॉलेशन (All Fixes)
print("⏳ इंजन तैयार हो रहा है...")
!pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
print("✅ सेटअप पूरा हुआ!")

In [ ]:
# @title 💤 Step 2: Anti-Sleep Mode
from IPython.display import display, Javascript
display(Javascript('''
    function ClickConnect(){ document.querySelector("colab-connect-button").click() }
    setInterval(ClickConnect, 60000)
'''))
print("🚀 Anti-Sleep सक्रिय है।")

In [ ]:
# @title 🚀 Step 3: app.py (Realistic Voice & All Controls)
import os

app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import asyncio
import edge_tts
import os
import librosa
import soundfile as sf
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📥 Loading Realistic Engine on {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def process_voice(audio_path, speed, pitch):
    y, sr = librosa.load(audio_path)
    # 1. Silence Remover
    y, _ = librosa.effects.trim(y, top_db=25)
    # 2. Speed Control
    if speed != 1.0:
        y = librosa.effects.time_stretch(y, rate=speed)
    # 3. Pitch Control
    if pitch != 0:
        y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    final_path = "output_final.wav"
    sf.write(final_path, y, sr)
    return final_path

async def fast_tts(text, voice, speed, pitch):
    output = 'fast_voice.mp3'
    rate = f"{speed:+}%"
    p = f"{pitch:+}Hz"
    communicate = edge_tts.Communicate(text, voice, rate=rate, pitch=p)
    await communicate.save(output)
    return output

def clone_voice(text, audio_sample, speed_val, pitch_val):
    if audio_sample is None:
        return None
    output_path = 'temp_clone.wav'
    # Realistic Cloning Logic
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language='hi', file_path=output_path)
    return process_voice(output_path, speed_val, pitch_val)

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.1')
    
    with gr.Tabs():
        with gr.TabItem('🧬 Realistic Voice Cloning'):
            gr.Markdown('### अपनी आवाज़ अपलोड करें और हुबहू आवाज़ पाएँ')
            with gr.Row():
                with gr.Column():
                    input_text = gr.Textbox(label='हिंदी टेक्स्ट लिखें', lines=5)
                    # Upload Feature Fixed
sample = gr.Audio(label='वॉइस सैंपल अपलोड करें (MP3/WAV)', type='filepath')
                    with gr.Row():
                        speed_slider = gr.Slider(0.5, 2.0, 1.0, step=0.1, label="Speed (रफ़्तार)")
                        pitch_slider = gr.Slider(-10, 10, 0, step=1, label="Pitch (भारीपन)")
                    btn_clone = gr.Button('Realistic Voice Generate 🚀', variant='primary')
                output_clone = gr.Audio(label='तैयार आवाज़')
            btn_clone.click(clone_voice, [input_text, sample, speed_slider, pitch_slider], output_clone)
            
        with gr.TabItem('⚡ Fast Standard TTS'):
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='टेक्स्ट', lines=5)
                    v_drop = gr.Dropdown(choices=['hi-IN-MadhurNeural', 'hi-IN-SwaraNeural'], label='आवाज़', value='hi-IN-MadhurNeural')
                    spd = gr.Slider(-50, 50, 0, label="Speed %")
                    ptc = gr.Slider(-20, 20, 0, label="Pitch")
                    btn_fast = gr.Button('Quick Generate')
                output_fast = gr.Audio(label='फास्ट ऑडियो')
            btn_fast.click(lambda t, v, s, p: asyncio.run(fast_tts(t, v, s, p)), [t_text, v_drop, spd, ptc], output_fast)

demo.launch(share=True)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("✅ app.py अपडेट हो गया। अब सभी बटन और अपलोड फीचर काम करेंगे।")
!python app.py